# Conditional Workflows in LangGraph

## 1. What is a Conditional Workflow?

A **Conditional Workflow** in LangGraph is a workflow where the **next step is selected dynamically based on the current state or some condition**.

### Simple Definition

> A Conditional Workflow allows a LangGraph application to make a decision and choose different paths based on data, state, or the result of a previous node.

### Basic Structure

                    ┌→ Path A → Node A ──┐
    START → Decision ┤                    ├→ END
                    └→ Path B → Node B ──┘

Instead of always following:

    START → A → B → C → END

the graph can decide:

    START
      ↓
    Decision
      ↓
    ┌───────────────┐
    ↓               ↓
    Condition A     Condition B
    ↓               ↓
    Path A          Path B


---

# 2. Why Do We Need Conditional Workflows?

Not every workflow should follow the same path.

For example, an AI customer-support system may receive:

    User Query

If the query is about:

    Billing → Billing Agent

If it is about:

    Technical Issue → Technical Agent

If it is about:

    General Question → General Agent

So:

                    ┌→ Billing
                    │
    User → Decision ├→ Technical
                    │
                    └→ General

The path depends on the input.

This is the main purpose of conditional workflows.


---

# 3. Sequential vs Parallel vs Conditional

## Sequential Workflow

The path is fixed.

    START
      ↓
      A
      ↓
      B
      ↓
      C
      ↓
     END

Meaning:

> Always execute A → B → C.


## Parallel Workflow

Multiple independent paths execute.

    START
      ↓
    ┌───┬───┬───┐
    ↓   ↓   ↓
    A   B   C
    └───┼───┘
        ↓
       END

Meaning:

> Execute multiple independent branches.


## Conditional Workflow

The system chooses a path.

    START
      ↓
    Decision
     /    \
    ↓      ↓
    A      B
    \      /
     ↓    ↓
      END

Meaning:

> Choose the appropriate branch based on a condition.


---

# 4. Core Idea

The most important idea is:

> **The current state determines what happens next.**

For example:

    State:
    {
        "score": 85
    }

Decision:

    if score >= 50:
        → Pass
    else:
        → Fail

Graph:

              ┌→ Pass → END
    Decision ─┤
              └→ Fail → END


---

# 5. Conditional Edges in LangGraph

LangGraph uses **conditional edges** to implement dynamic routing.

A normal edge says:

    A → B

Meaning:

> Whenever A finishes, go to B.

A conditional edge says:

    A → ?

Meaning:

> After A finishes, evaluate a routing function and decide where to go.


Conceptually:

    A
    ↓
    Router
    ↓
    ┌───────┬───────┐
    ↓       ↓       ↓
    B       C       D


---

# 6. Normal Edge vs Conditional Edge

### Normal Edge

    graph.add_edge("A", "B")

Means:

    A → B

The transition is fixed.


### Conditional Edge

Conceptually:

    graph.add_conditional_edges(
        "A",
        router
    )

Meaning:

    A
    ↓
    router()
    ↓
    dynamically choose next node


This is one of the most important LangGraph concepts.


---

# 7. Basic Conditional Workflow

Example:

    START
      ↓
    classify
      ↓
    ┌─────────────┐
    ↓             ↓
   math         general
    ↓             ↓
    A             B
    └──────┬──────┘
           ↓
          END


The `classify` node determines the category.

For example:

    Input:
    "What is 25 × 10?"

Classification:

    "math"

Therefore:

    classify → math_node → END


If input is:

    "What is LangGraph?"

Classification:

    "general"

Therefore:

    classify → general_node → END


---

# 8. State in Conditional Workflows

State stores the information used to make decisions.

Example:

    class State(TypedDict):
        user_input: str
        category: str
        response: str

Initial state:

    {
        "user_input": "What is 25 * 10?"
    }

Classifier updates:

    {
        "category": "math"
    }

Router reads:

    state["category"]

and chooses:

    math_node


---

# 9. The Router Function

A **router function** determines which path should be taken.

Conceptually:

    def router(state):
        if state["category"] == "math":
            return "math"

        elif state["category"] == "technical":
            return "technical"

        else:
            return "general"


The router does not necessarily perform the actual work.

Its primary responsibility is:

> Decide where execution should go next.


---

# 10. Router vs Node

These two concepts are easy to confuse.

### Node

A node performs work.

Examples:

    summarize()
    retrieve_documents()
    call_llm()
    execute_sql()


### Router

A router decides where to go.

Example:

    if query_type == "sql":
        → sql_node

    if query_type == "search":
        → search_node

So:

    Node = Do the work

    Router = Decide the path


---

# 11. Conceptual Code

A simple conditional workflow:

    from typing import TypedDict
    from langgraph.graph import StateGraph, START, END


    class State(TypedDict):
        number: int
        result: str


    def check_number(state: State):
        return {}


    def positive_node(state: State):
        return {
            "result": "Number is positive"
        }


    def negative_node(state: State):
        return {
            "result": "Number is negative"
        }


    def router(state: State):
        if state["number"] >= 0:
            return "positive"

        return "negative"


    graph = StateGraph(State)

    graph.add_node("check_number", check_number)
    graph.add_node("positive", positive_node)
    graph.add_node("negative", negative_node)

    graph.add_edge(START, "check_number")

    graph.add_conditional_edges(
        "check_number",
        router,
        {
            "positive": "positive",
            "negative": "negative"
        }
    )

    graph.add_edge("positive", END)
    graph.add_edge("negative", END)

    app = graph.compile()

    result = app.invoke({
        "number": 10
    })


The important part is:

    graph.add_conditional_edges(
        "check_number",
        router,
        {
            "positive": "positive",
            "negative": "negative"
        }
    )


---

# 12. How Conditional Routing Works

Execution:

    Initial State
         ↓
    check_number
         ↓
       router
         ↓
    ┌────┴────┐
    ↓         ↓
 positive   negative
    ↓         ↓
   END       END


For:

    number = 10

Router returns:

    "positive"

Flow:

    START
      ↓
    check_number
      ↓
    router
      ↓
    positive
      ↓
    END


For:

    number = -5

Flow:

    START
      ↓
    check_number
      ↓
    router
      ↓
    negative
      ↓
    END


---

# 13. Conditional Routing with Multiple Paths

Conditional workflows are not limited to two branches.

Example:

    START
      ↓
    classify_query
          ↓
       Router
     /    |     \
    ↓     ↓      ↓
  SQL   Search  General
    ↓     ↓      ↓
    └─────┼──────┘
          ↓
         END


Router:

    def router(state):
        query_type = state["query_type"]

        if query_type == "sql":
            return "sql"

        if query_type == "search":
            return "search"

        return "general"


This is often called **multi-way routing**.


---

# 14. Conditional Workflow with LLM

Conditional routing becomes very powerful when an LLM performs classification.

Example:

    User Query
        ↓
       LLM
        ↓
    Classify Query
        ↓
    ┌───────┬────────┬────────┐
    ↓       ↓        ↓
   SQL     RAG      General
    ↓       ↓        ↓
    └───────┼────────┘
            ↓
          Answer


Example inputs:

    "Show me total sales by month."

→ SQL


    "What does our employee handbook say about leave?"

→ RAG


    "Explain machine learning."

→ General LLM


---

# 15. Conditional Workflow in AI Data Analyst

Suppose we are building an AI Data Analyst.

User asks:

    "Show total revenue by month."

The system first determines:

    Is this a data question?

Yes.

Then:

    SQL generation
        ↓
    SQL validation
        ↓
    SQL execution
        ↓
    Result analysis


But suppose the user asks:

    "Explain what machine learning is."

The system can route to:

    General LLM
        ↓
      Answer


Architecture:

                    ┌→ SQL Agent → Execute → Analyze ──┐
    User → Router ──┤                                   ├→ Answer
                    └→ General LLM ────────────────────┘


---

# 16. Conditional Workflow in RAG

A RAG system does not necessarily need retrieval for every query.

Example:

    User Query
        ↓
    Need Retrieval?
       /    \
     Yes     No
      ↓       ↓
   Retrieve  LLM
      ↓       ↓
      └───┬───┘
          ↓
        Answer


Example:

    "What is our refund policy?"

→ Retrieve documents.


But:

    "What is 2 + 2?"

→ Directly answer.


This is sometimes called **conditional RAG** or **adaptive routing** depending on the exact architecture.


---

# 17. Conditional RAG Architecture

A more complete architecture:

    User Query
        ↓
    Query Classifier
        ↓
    ┌──────────────┐
    ↓              ↓
    RAG          Direct LLM
    ↓              ↓
 Retriever          │
    ↓              │
 Context            │
    ↓              │
    └───────┬──────┘
            ↓
        Generate Answer
            ↓
           END


This avoids unnecessary retrieval when it is not needed.


---

# 18. Conditional Workflow with Validation

Another common pattern:

    START
      ↓
    Generate
      ↓
    Validate
      ↓
    ┌──────────────┐
    ↓              ↓
   Valid        Invalid
    ↓              ↓
   END           Retry
                   ↓
                Generate


This introduces a loop.

The router checks:

    Is output valid?

If yes:

    → END

If no:

    → Generate again


---

# 19. Conditional Workflow with Loops

Conditional edges can create loops.

Example:

             ┌───────────────┐
             │               ↓
    START → Generate → Validate
                         /   \
                        /     \
                     Invalid  Valid
                       ↓        ↓
                    Generate   END


This means:

    Generate
       ↓
    Validate
       ↓
    Valid?
     /   \
   No     Yes
   ↓       ↓
 Generate  END


This is extremely useful for Agentic AI systems.


---

# 20. Conditional Loop Example

Suppose an LLM generates SQL.

Workflow:

    Generate SQL
        ↓
    Validate SQL
        ↓
    Valid?
     /    \
   No      Yes
   ↓        ↓
 Retry    Execute
   ↓        ↓
Generate  Analyze
            ↓
           END


The system does not blindly continue.

It checks the current state and chooses the next action.


---

# 21. Conditional Workflow in Agentic AI

Agentic systems depend heavily on conditional routing.

Basic agent loop:

    START
      ↓
    Agent
      ↓
    Decision
     /    \
    ↓      ↓
   Tool   Final
    ↓      ↓
 Observe   END
    ↓
   Agent


The agent decides:

> Do I need another tool call, or can I answer?

This is conditional routing.


---

# 22. Agent Tool-Calling Example

Suppose the user asks:

    "What is the weather in Mumbai?"

Agent:

    Need external information?
          ↓
         Yes
          ↓
      Weather Tool
          ↓
       Observe
          ↓
        Agent
          ↓
       Answer


But if user asks:

    "What is Python?"

Agent may decide:

    No external tool required
          ↓
       Final Answer


Architecture:

                    ┌→ Tool → Observe ─┐
    Agent → Router ─┤                  ├→ Agent
                    └→ Final → END     │
                                       │
                                       └── loop


---

# 23. Conditional Workflow vs Agentic Decision Making

They are related but not identical.

### Conditional Workflow

Developer defines the possible conditions/routes.

Example:

    if type == "SQL":
        → SQL node

    if type == "RAG":
        → RAG node


### Agentic Decision Making

The LLM can determine what action is appropriate.

Example:

    "I need to search the database."

    → SQL tool


The graph still controls what transitions are allowed.


---

# 24. Why LangGraph Is Useful for Conditional Workflows

LangGraph provides explicit control over:

- State
- Nodes
- Edges
- Conditional edges
- Loops
- Routing
- Persistence
- Human approval
- Error handling

Instead of hiding the workflow inside one large function, the decision structure is represented explicitly as a graph.

This makes complex workflows easier to understand and control.


---

# 25. Conditional Workflow Architecture

A general architecture:

    ┌─────────────────────┐
    │      Input          │
    └──────────┬──────────┘
               ↓
    ┌─────────────────────┐
    │    Processing       │
    └──────────┬──────────┘
               ↓
    ┌─────────────────────┐
    │      Router         │
    └──────────┬──────────┘
               ↓
         ┌─────┼─────┐
         ↓     ↓     ↓
       Path A Path B Path C
         ↓     ↓     ↓
         └─────┼─────┘
               ↓
          Aggregation
               ↓
              END


---

# 26. Conditional Routing Based on State

The condition can be based on almost any relevant state value.

Examples:

### User type

    state["user_type"]

### Query type

    state["query_type"]

### Confidence

    state["confidence"]

### Validation status

    state["is_valid"]

### Tool result

    state["tool_success"]

### Retrieved documents

    state["documents"]

### Number of attempts

    state["attempts"]


Example:

    if state["confidence"] > 0.8:
        → answer

    else:
        → human_review


---

# 27. Conditional Human-in-the-Loop

Conditional routing can decide when human approval is necessary.

Example:

    AI generates response
           ↓
       Confidence
        /       \
    High         Low
     ↓            ↓
   Send       Human Review
                 ↓
              Approved?
              /      \
            Yes       No
             ↓         ↓
            Send      Reject
             ↓
            END


This is useful when an AI system should not make every decision autonomously.


---

# 28. Conditional Workflow with Confidence

Suppose an AI classifier returns:

    confidence = 0.92

Rule:

    confidence >= 0.80
        → automatic response

Otherwise:

    confidence < 0.80
        → human review


Architecture:

                  ┌→ Auto Response → END
                  │
    Classify → Router
                  │
                  └→ Human Review → END


This is a common production pattern.


---

# 29. Conditional Workflow with Error Handling

Example:

    Call API
       ↓
    Success?
     /    \
   Yes     No
    ↓       ↓
 Continue  Retry
            ↓
          Success?
          /     \
        Yes      No
         ↓        ↓
      Continue  Fallback


Conditional routing allows the system to recover from failures.


---

# 30. Retry Limits

A retry loop should usually have a maximum number of attempts.

State:

    attempts = 0

After failure:

    attempts += 1

Router:

    if attempts < 3:
        → retry

    else:
        → fallback


Architecture:

                  ┌→ Retry ──────┐
                  │              │
    Execute → Check              │
                  │              ↓
                  └→ Fallback ← attempts >= 3
                         ↓
                        END


This prevents infinite loops.


---

# 31. Conditional Workflow and Guardrails

Conditional routing can enforce safety and business rules.

Example:

    User Input
        ↓
    Safety Check
        ↓
    ┌──────────────┐
    ↓              ↓
   Safe         Unsafe
    ↓              ↓
   Agent       Refuse/Handle
    ↓              ↓
   END            END


The graph controls which operations are allowed after validation.


---

# 32. Conditional Workflow in AI Travel Planner

For an AI Travel Planner:

    User Request
         ↓
    Extract Destination
         ↓
    Destination Valid?
       /       \
     Yes        No
      ↓          ↓
   Research    Ask User
      ↓
    Budget?
    /     \
  Yes      No
   ↓        ↓
Budget     Default
Research   Budget
   ↓        ↓
   └───┬────┘
       ↓
    Itinerary
       ↓
      END


Another possibility:

    User Query
        ↓
    Trip Type
      / | \
     ↓  ↓  ↓
   Budget Luxury Adventure
     ↓  ↓  ↓
     └──┼──┘
        ↓
      Planner


---

# 33. Conditional Workflow in Multi-Agent Systems

A supervisor can route work to specialized agents.

    User
     ↓
    Supervisor
     ↓
    ┌────────┬────────┬────────┐
    ↓        ↓        ↓
 Research  Coding   Data
 Agent     Agent    Agent
    ↓        ↓        ↓
    └────────┼────────┘
             ↓
         Supervisor
             ↓
            END


The supervisor makes a routing decision.

For example:

    "Find information about competitors."

→ Research Agent


    "Debug this Python code."

→ Coding Agent


    "Analyze this CSV."

→ Data Agent


---

# 34. Conditional vs Parallel

These are often combined.

Example:

    User
      ↓
    Classifier
      ↓
    Need research?
      ↓
     Yes
      ↓
    ┌────────┬────────┬────────┐
    ↓        ↓        ↓
   Web      RAG      Database
    └────────┼────────┘
             ↓
          Combine
             ↓
            END


Here:

1. Conditional routing decides whether research is needed.
2. Parallel workflow executes multiple research tasks.


So LangGraph workflows can combine multiple patterns.


---

# 35. Conditional + Sequential

Example:

    START
      ↓
    Validate
      ↓
    Valid?
     /   \
   Yes    No
    ↓      ↓
 Process  Fix
    ↓      ↓
    END   Validate

This combines:

    Conditional
    +
    Sequential
    +
    Loop


---

# 36. Conditional + Parallel + Sequential

A more realistic AI system may contain all three.

    START
      ↓
    Classify
      ↓
    Need Research?
      /       \
    No         Yes
    ↓           ↓
  Answer    ┌───┬───┬───┐
            ↓   ↓   ↓
           Web RAG DB
            └───┼───┘
                ↓
             Combine
                ↓
              Answer
                ↓
               END


This is why LangGraph becomes useful for complex AI systems.


---

# 37. Benefits of Conditional Workflows

### 1. Dynamic behavior

The workflow can react to different inputs.

### 2. Better efficiency

Unnecessary steps can be skipped.

### 3. Better control

Developers explicitly define allowed paths.

### 4. Easier error recovery

Failures can be routed to retry/fallback nodes.

### 5. Better personalization

Different users/requests can follow different paths.

### 6. Agentic behavior

Conditional routing is a core building block of agents.


---

# 38. Limitations / Challenges

Conditional workflows introduce additional complexity.

### 1. More branches

More paths mean more code and testing.

### 2. Incorrect routing

A bad classifier/router can send execution down the wrong path.

### 3. State management

Routing depends on correct state values.

### 4. Infinite loops

Poorly designed conditions can create endless cycles.

### 5. Debugging

It can be harder to determine why a particular path was selected.

### 6. LLM-based routing can be unreliable

If an LLM decides the route, structured output and validation are often important.


---

# 39. Best Practices

## 1. Keep router logic simple

Prefer:

    if/elif/else

or a clear mapping.

Avoid putting large business logic inside the router.


## 2. Separate routing from execution

Use:

    Router → Decide

    Node → Execute


## 3. Use explicit route names

For example:

    "sql"
    "rag"
    "general"

rather than unclear values.


## 4. Validate LLM-generated routes

If an LLM produces a category, ensure it belongs to the allowed set.


## 5. Handle unexpected values

Always have a fallback/default route where appropriate.


## 6. Limit loops

Use an attempt counter or another termination condition.


## 7. Keep state intentional

Only store information required by downstream nodes.


---

# 40. Common Mistakes

## Mistake 1: Confusing node and router

Remember:

    Node = performs work

    Router = chooses path


## Mistake 2: Using normal edges for dynamic behavior

If the destination depends on state, use conditional routing.


## Mistake 3: Forgetting a fallback

Unexpected state values can cause routing failures.


## Mistake 4: Infinite loops

Always ensure there is a condition that eventually reaches END or another terminating state.


## Mistake 5: Letting an LLM freely control the graph

Production systems should constrain possible routes.


## Mistake 6: Making routing overly complicated

A router should generally decide the next path, not perform the entire workflow.


---

# 41. Conditional Routing with a Mapping

A clean pattern is:

    router_result = "sql"

Then map it:

    {
        "sql": "sql_node",
        "rag": "rag_node",
        "general": "general_node"
    }


Conceptually:

    Router Output
         ↓
    ┌────┼────┐
    ↓    ↓    ↓
   sql  rag general
    ↓    ↓    ↓
   Node Node Node


This separates:

    Decision value

from:

    Actual graph node.


---

# 42. Conditional Workflow Execution Model

Remember this execution model:

    Initial State
         ↓
        Node
         ↓
       Router
         ↓
    Evaluate Condition
         ↓
    Select Destination
         ↓
    Next Node
         ↓
    State Update
         ↓
    Router / Next Node
         ↓
       ...
         ↓
        END


The important point:

> **The graph does not necessarily have one fixed path. The path can change according to state.**


---

# 43. Conditional Workflow vs If Statement

They are conceptually related but operate at different levels.

Python:

    if condition:
        do_a()
    else:
        do_b()


LangGraph:

    Node
      ↓
    Conditional Edge
      ↓
    ┌───────┬───────┐
    ↓       ↓
   Node A  Node B


The LangGraph version represents the decision as part of the **workflow graph**.

This makes the workflow:

- Visualizable
- Modular
- Stateful
- Composable
- Easier to extend


---

# 44. Conditional Workflow vs Traditional Workflow

Traditional application:

    if query_type == "sql":
        run_sql()

    elif query_type == "rag":
        run_rag()

    else:
        run_llm()


LangGraph:

    START
      ↓
    Classifier
      ↓
    Conditional Edge
     /      |      \
    ↓       ↓       ↓
   SQL     RAG     LLM
    ↓       ↓       ↓
    └───────┼───────┘
            ↓
           END


The logic is similar, but LangGraph makes the workflow structure explicit.


---

# 45. Interview Questions

### Q1. What is a conditional workflow in LangGraph?

A workflow where the next node/path is dynamically selected based on the current state or a condition.

### Q2. What is a conditional edge?

A graph transition whose destination is determined dynamically by a routing function.

### Q3. What is a router?

A function or component that examines state and determines the next path.

### Q4. What is the difference between a node and a router?

A node performs a task; a router determines where execution should go next.

### Q5. Can conditional workflows have more than two branches?

Yes. They can route to multiple possible nodes.

### Q6. Can conditional edges create loops?

Yes. A conditional edge can route execution back to an earlier node.

### Q7. How do you prevent infinite loops?

Use explicit termination conditions, retry limits, attempt counters, or state-based stopping criteria.

### Q8. Can an LLM be used as a router?

Yes. An LLM can classify input or determine the next action, but its output should generally be constrained and validated.

### Q9. Can conditional and parallel workflows be combined?

Yes.

Example:

    Conditional Router
          ↓
    Parallel Research
          ↓
       Combine


### Q10. Why are conditional workflows important for Agentic AI?

Agents need to decide what action to take next based on the current state and observations. Conditional routing provides the workflow-level mechanism for those decisions.


---

# 46. Quick Revision

### Definition

> Conditional Workflow = A workflow where the next step is dynamically selected based on state or conditions.

### Core Components

- State
- Node
- Router
- Conditional Edge
- Branches
- State Updates
- Termination condition

### Architecture

    START
      ↓
    Node
      ↓
    Router
      ↓
    ┌───────┬───────┬───────┐
    ↓       ↓       ↓
   A       B       C
    ↓       ↓       ↓
    └───────┼───────┘
            ↓
           END


### Key Terms

    Node   → Does work

    Router → Makes decision

    Edge   → Connects nodes

    Conditional Edge
           → Chooses destination dynamically


---

# 47. Three LangGraph Workflow Patterns So Far

## 1. Sequential

    A → B → C

> Fixed order.


## 2. Parallel

    A ──┐
    B ──┼→ D
    C ──┘

> Independent tasks execute in parallel.


## 3. Conditional

    A
    ↓
    Router
    /   \
   B     C

> Choose the next path dynamically.


---

# 48. Combined Mental Model

Real-world LangGraph applications often combine all three:

                    ┌→ Task A ──┐
                    │           │
    START → Router ─┼→ Task B ──┼→ Combine → Validate
                    │           │                ↓
                    └→ Task C ──┘             Valid?
                                               /    \
                                             Yes     No
                                              ↓       ↓
                                             END    Retry
                                                     ↓
                                                  Router

This single architecture contains:

    Conditional routing
          +
    Parallel execution
          +
    Sequential processing
          +
    Validation
          +
    Loop/retry


---

# 49. Final Mental Model

Think of a traffic intersection.

    You arrive at an intersection
            ↓
       Traffic Signal
            ↓
       ┌────┼────┐
       ↓    ↓    ↓
      Left Straight Right

The intersection decides which road you take.

Similarly, in LangGraph:

    State
      ↓
    Router
      ↓
    Conditional Edge
      ↓
    ┌────┼────┐
    ↓    ↓    ↓
    A    B    C

The router examines the state and determines the next node.

### One-line memory trick

> **Sequential = Follow the fixed path.**
>
> **Parallel = Execute multiple independent paths.**
>
> **Conditional = Choose the path based on the current state.**

### LangGraph Formula

    Conditional Workflow
    =
    State
    +
    Decision/Router
    +
    Conditional Edge
    +
    Dynamic Path
    +
    State Update
    +
    Next Node
    +
    END / Loop

### Agentic AI Connection

    Goal
      ↓
    Agent
      ↓
    Observe State
      ↓
    Decide
      ↓
    Conditional Route
      ↓
    Tool / Node
      ↓
    Observe Result
      ↓
    Decide Again
      ↓
    END

Therefore:

> **Conditional workflows are one of the fundamental building blocks of Agentic AI because an agent must dynamically decide what should happen next.**